In [2]:
# GPU NVIDIA (CUDA 12.1). Jika CPU-only, ganti baris torch sesuai instruksi sebelumnya.
!pip -q install --upgrade pip
!pip -q install "torch==2.3.1+cu121" --index-url https://download.pytorch.org/whl/cu121
!pip -q install "transformers==4.44.2" "accelerate==0.33.0" "datasets==2.20.0" "evaluate==0.4.1"
!pip -q install "scikit-learn==1.5.2" "pandas==2.2.2" "pyarrow==16.1.0" "ipykernel" sentencepiece==0.1.99
# (opsional) progress bar di notebook
!pip -q install ipywidgets==8.1.2 jupyterlab_widgets

In [2]:
import sys, torch, transformers, datasets, accelerate
print("Python:", sys.executable)
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| is_cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__, "| datasets:", datasets.__version__, "| accelerate:", accelerate.__version__)

Python: D:\.Portofolio\Coding\social-sentiment\.venv\Scripts\python.exe
torch: 2.3.1+cu121 | cuda: 12.1 | is_cuda: True
transformers: 4.44.2 | datasets: 2.20.0 | accelerate: 0.33.0


In [3]:
from datasets import load_dataset

# English: TweetEval sentiment (0=neg,1=neu,2=pos)
ds_en = load_dataset("cardiffnlp/tweet_eval", "sentiment")

# Indonesian: IndoNLU SmSA (perlu trust_remote_code) → fallback ke NusaX-senti jika gagal
try:
    ds_id = load_dataset("indonlp/indonlu", "smsa", trust_remote_code=True)
    src_id = "indonlu-smsa"
except Exception as e:
    print("⚠️ Gagal load IndoNLU SmSA, fallback ke NusaX-senti (id). Error:", e)
    ds_id = load_dataset("indonlp/NusaX-senti", "id")
    src_id = "nusax-senti"

ds_en, ds_id, src_id

(DatasetDict({
     train: Dataset({
         features: ['text', 'label'],
         num_rows: 45615
     })
     test: Dataset({
         features: ['text', 'label'],
         num_rows: 12284
     })
     validation: Dataset({
         features: ['text', 'label'],
         num_rows: 2000
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['text', 'label'],
         num_rows: 11000
     })
     validation: Dataset({
         features: ['text', 'label'],
         num_rows: 1260
     })
     test: Dataset({
         features: ['text', 'label'],
         num_rows: 500
     })
 }),
 'indonlu-smsa')

In [4]:
import pandas as pd
from datasets import Dataset, DatasetDict

# EN (TweetEval): sudah int 0/1/2
def map_en(x):
    return {"text": x["text"], "label": int(x["label"]), "lang": "en"}

en_splits = {k: ds_en[k].map(map_en, remove_columns=ds_en[k].column_names) for k in ds_en.keys()}

# ID: bisa ClassLabel int atau string tergantung sumber
try:
    id_label_names = ds_id["train"].features["label"].names
except Exception:
    id_label_names = None

def map_id(x):
    # Ambil nama label
    if id_label_names and isinstance(x["label"], int):
        name = id_label_names[x["label"]]
    else:
        name = str(x["label"])
    name = name.lower()

    mapping = {"negative": 0, "neutral": 1, "positive": 2}
    if name not in mapping:
        # fallback: bila sudah berupa int konsisten 0/1/2
        if isinstance(x["label"], int):
            return {"text": x["text"], "label": int(x["label"]), "lang": "id"}
        raise ValueError(f"Label tidak dikenali: {name}")
    return {"text": x["text"], "label": mapping[name], "lang": "id"}

id_splits = {k: ds_id[k].map(map_id, remove_columns=ds_id[k].column_names) for k in ds_id.keys()}

def concat_if_exists(split):
    parts = []
    if split in en_splits: parts.append(en_splits[split])
    if split in id_splits: parts.append(id_splits[split])
    if len(parts) == 1:
        return parts[0]
    return Dataset.from_pandas(pd.concat([p.to_pandas() for p in parts], ignore_index=True))

merged = DatasetDict({
    "train": concat_if_exists("train"),
    "validation": concat_if_exists("validation"),
    "test": concat_if_exists("test"),
})
merged

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'lang'],
        num_rows: 56615
    })
    validation: Dataset({
        features: ['text', 'label', 'lang'],
        num_rows: 3260
    })
    test: Dataset({
        features: ['text', 'label', 'lang'],
        num_rows: 12784
    })
})

In [5]:
for split in merged:
    df = merged[split].to_pandas()
    print(f"=== {split.upper()} ===")
    print("Rows:", len(df))
    print("Lang counts:\n", df["lang"].value_counts(), "\n")
    print("Label counts (0=neg,1=neu,2=pos):\n", df["label"].value_counts(), "\n")
    display(df.head(3))

=== TRAIN ===
Rows: 56615
Lang counts:
 lang
en    45615
id    11000
Name: count, dtype: int64 

Label counts (0=neg,1=neu,2=pos):
 label
2    24265
1    21821
0    10529
Name: count, dtype: int64 



,text,label,lang
0,"""QT @user In the original draft of the 7th boo...",2,en
1,"""Ben Smith / Smith (concussion) remains out of...",1,en
2,Sorry bout the stream last night I crashed out...,1,en


=== VALIDATION ===
Rows: 3260
Lang counts:
 lang
en    2000
id    1260
Name: count, dtype: int64 

Label counts (0=neg,1=neu,2=pos):
 label
2    1554
1    1000
0     706
Name: count, dtype: int64 



,text,label,lang
0,Dark Souls 3 April Launch Date Confirmed With ...,1,en
1,"""National hot dog day, national tequila day, t...",2,en
2,When girls become bandwagon fans of the Packer...,0,en


=== TEST ===
Rows: 12784
Lang counts:
 lang
en    12284
id      500
Name: count, dtype: int64 

Label counts (0=neg,1=neu,2=pos):
 label
1    6025
0    4176
2    2583
Name: count, dtype: int64 



,text,label,lang
0,@user @user what do these '1/2 naked pics' hav...,1,en
1,OH: “I had a blue penis while I was this” [pla...,1,en
2,"@user @user That's coming, but I think the vic...",1,en


In [6]:
from transformers import AutoTokenizer

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

def load_tokenizer_robust(model_name):
    # 1) coba fast
    try:
        print("Trying FAST tokenizer...")
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        print("Loaded FAST tokenizer.")
        return tok
    except Exception as e1:
        print("FAST tokenizer failed:", e1)

    # 2) coba slow (butuh sentencepiece)
    try:
        import sentencepiece  # memastikan modul ada
        print("Trying SLOW tokenizer (SentencePiece)...")
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        print("Loaded SLOW tokenizer.")
        return tok
    except Exception as e2:
        print("SLOW tokenizer failed:", e2)

    # 3) fallback ke base vocab yang kompatibel
    try:
        print("FALLBACK to 'xlm-roberta-base' tokenizer...")
        tok = AutoTokenizer.from_pretrained("xlm-roberta-base", use_fast=False)
        print("Loaded fallback tokenizer (xlm-roberta-base).")
        return tok
    except Exception as e3:
        print("Fallback tokenizer also failed:", e3)
        raise

tokenizer = load_tokenizer_robust(MODEL_NAME)

Trying FAST tokenizer...


D:\.Portofolio\Coding\social-sentiment\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


FAST tokenizer failed: 
XLMRobertaConverter requires the protobuf library but it was not found in your environment. Checkout the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Trying SLOW tokenizer (SentencePiece)...
Loaded SLOW tokenizer.


In [7]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification

num_labels = 3
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

# (opsional) hemat VRAM
# model.gradient_checkpointing_enable()

In [8]:
# === (Re)TOKENIZATION ===
from transformers import DataCollatorWithPadding

# pilih dataset yang tersedia
try:
    _ds = merged_clean   # ada kalau kamu sudah jalankan cleaning
except NameError:
    _ds = merged         # fallback ke dataset gabungan tanpa cleaning

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=160)

tokd = _ds.map(tok, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# cek cepat
print({k: tokd[k].num_rows for k in tokd})

Map:   0%|          | 0/56615 [00:00<?, ? examples/s]

Map:   0%|          | 0/3260 [00:00<?, ? examples/s]

Map:   0%|          | 0/12784 [00:00<?, ? examples/s]

{'train': 56615, 'validation': 3260, 'test': 12784}


In [9]:
import torch
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

args = TrainingArguments(
    output_dir="artifacts/xlmr-sentiment",
    learning_rate=1e-5,                 # lebih kecil karena start dari checkpoint tugas-semua
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,                  # early stopping akan hentikan lebih awal jika perlu
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    warmup_ratio=0.1,
    logging_steps=100,
    seed=42,
    fp16=torch.cuda.is_available(),
    label_smoothing_factor=0.05
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokd["train"],
    eval_dataset=tokd["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

D:\.Portofolio\Coding\social-sentiment\.venv\Lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [10]:
# Train
train_result = trainer.train()

# Evaluate
val_metrics = trainer.evaluate(tokd["validation"])
test_metrics = trainer.evaluate(tokd["test"])
print("Val:", val_metrics)
print("Test:", test_metrics)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.625200,0.507580,0.824847,0.814023
2,0.555800,0.516092,0.833742,0.818838
3,0.447800,0.513380,0.836810,0.825615
4,0.398800,0.550252,0.832822,0.820717
5,0.366600,0.562433,0.836810,0.825798


Val: {'eval_loss': 0.5624326467514038, 'eval_accuracy': 0.8368098159509203, 'eval_macro_f1': 0.8257981465770822, 'eval_runtime': 4.4281, 'eval_samples_per_second': 736.215, 'eval_steps_per_second': 23.035, 'epoch': 5.0}
Test: {'eval_loss': 0.8753738403320312, 'eval_accuracy': 0.7109668335419274, 'eval_macro_f1': 0.7137432356347512, 'eval_runtime': 14.6615, 'eval_samples_per_second': 871.945, 'eval_steps_per_second': 27.282, 'epoch': 5.0}


In [9]:
def clean_text(t: str):
    import re
    t = re.sub(r"http\S+|www\.\S+", "<URL>", t)
    t = re.sub(r"@\w+", "<USER>", t)
    return t.strip()

def add_clean(batch):
    return {"text": [clean_text(x) for x in batch["text"]]}

from datasets import DatasetDict
merged_clean = DatasetDict({
    split: merged[split].map(add_clean, batched=True)
    for split in merged.keys()
})

Map:   0%|          | 0/56615 [00:00<?, ? examples/s]

Map:   0%|          | 0/3260 [00:00<?, ? examples/s]

Map:   0%|          | 0/12784 [00:00<?, ? examples/s]

In [15]:
import numpy as np
import torch
from sklearn.metrics import classification_report

def eval_subset(dataset, lang_code, batch_size=32, max_len=128, use_fp16=True):
    """
    Evaluasi per-bahasa dengan batching kecil supaya tidak OOM.
    - batch_size: kecilkan jika masih OOM (coba 16/8).
    - max_len: 128 sudah cukup untuk komentar; lebih kecil = lebih hemat VRAM.
    - use_fp16: True jika GPU mendukung (hemat ~2x VRAM).
    """
    df = dataset.to_pandas()
    sub = df[df["lang"] == lang_code]
    if len(sub) == 0:
        print(f"No samples for lang={lang_code}")
        return

    texts = sub["text"].tolist()
    y_true = sub["label"].to_numpy()
    preds_all = []

    device = model.device
    model.eval()

    # optional: set fp16 autocast jika GPU
    autocast_ctx = torch.cuda.amp.autocast if (use_fp16 and torch.cuda.is_available()) else torch.no_grad

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            enc = tokenizer(
                batch_texts,
                truncation=True,
                max_length=max_len,
                padding=True,
                return_tensors="pt"
            ).to(device)

            # fp16 untuk hemat VRAM
            if use_fp16 and torch.cuda.is_available():
                with torch.cuda.amp.autocast(dtype=torch.float16):
                    logits = model(**enc).logits
            else:
                logits = model(**enc).logits

            preds = logits.argmax(dim=1).detach().cpu().numpy()
            preds_all.append(preds)

            # bebaskan memori batch
            del enc, logits
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    y_pred = np.concatenate(preds_all, axis=0)
    print(f"Lang={lang_code}  n={len(texts)}")
    print(classification_report(y_true, y_pred, digits=4, target_names=["neg","neu","pos"]))

In [18]:
ds_val = merged_clean["validation"] if "merged_clean" in globals() else merged["validation"]
ds_test = merged_clean["test"]       if "merged_clean" in globals() else merged["test"]

print("=== VALIDATION per language ===")
eval_subset(ds_val, "en", batch_size=32, max_len=128, use_fp16=True)
eval_subset(ds_val, "id", batch_size=32, max_len=128, use_fp16=True)

print("=== TEST per language ===")
eval_subset(ds_test, "en", batch_size=32, max_len=128, use_fp16=True)
eval_subset(ds_test, "id", batch_size=32, max_len=128, use_fp16=True)

=== VALIDATION per language ===
Lang=en  n=2000
              precision    recall  f1-score   support

         neg     0.5567    0.8814    0.6824       312
         neu     0.7391    0.6812    0.7090       869
         pos     0.8596    0.7399    0.7953       819

    accuracy                         0.7365      2000
   macro avg     0.7184    0.7675    0.7289      2000
weighted avg     0.7600    0.7365    0.7402      2000

Lang=id  n=1260
              precision    recall  f1-score   support

         neg     0.7780    0.9340    0.8489       394
         neu     0.4717    0.5725    0.5172       131
         pos     0.9538    0.8150    0.8789       735

    accuracy                         0.8270      1260
   macro avg     0.7345    0.7738    0.7484      1260
weighted avg     0.8487    0.8270    0.8319      1260

=== TEST per language ===
Lang=en  n=12284
              precision    recall  f1-score   support

         neg     0.6185    0.8615    0.7200      3972
         neu     0.749

In [13]:
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()